<a href="https://colab.research.google.com/github/codebytelabs/2025_pl_predictions/blob/main/StrategyINDMarketAIStraregy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install dhanhq requests

In [ ]:
CLIENT_ID = ''
ACCESS_TOKEN = ''

In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║         DHAN EMA CROSSOVER ALGO - RELIANCE NSE               ║
║         Strategy : 9 EMA x 15 EMA  |  Qty : 1               ║
╚══════════════════════════════════════════════════════════════╝

SETUP:
    pip install dhanhq requests

CONFIGURE:
    Set CLIENT_ID and ACCESS_TOKEN below before running.
"""

import time
import random
import requests
from datetime import datetime

# ─────────────────────────────────────────────
#  DHAN CREDENTIALS  ← fill these in
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────
RELIANCE_SECURITY_ID = "1333"          # Dhan security ID for RELIANCE
EXCHANGE_SEGMENT     = "NSE_EQ"
PRODUCT_TYPE         = "CNC"           # CNC for delivery | INTRADAY for intraday
ORDER_TYPE           = "MARKET"
QUANTITY             = 1

BASE_URL = "https://api.dhan.co"

HEADERS = {
    "access-token": ACCESS_TOKEN,
    "client-id": CLIENT_ID,
    "Content-Type": "application/json",
}

# ─────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────

def ts():
    """Current timestamp string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def log(tag: str, msg: str):
    colour = {
        "INFO" : "\033[96m",   # cyan
        "ORDER": "\033[92m",   # green
        "WARN" : "\033[93m",   # yellow
        "ERROR": "\033[91m",   # red
        "ALGO" : "\033[95m",   # magenta
        "CLOSE": "\033[33m",   # orange-ish
    }.get(tag, "\033[0m")
    reset = "\033[0m"
    print(f"{colour}[{ts()}] [{tag}]{reset}  {msg}")

def separator(char="─", n=65):
    print("\033[90m" + char * n + "\033[0m")

# ─────────────────────────────────────────────
#  DHAN ORDER FUNCTIONS
# ─────────────────────────────────────────────

def place_market_order(transaction_type: str) -> dict:
    """
    Place a MARKET order on Dhan.
    transaction_type : "BUY" or "SELL"
    Returns the full API response dict.
    """
    payload = {
        "dhanClientId"    : CLIENT_ID,
        "transactionType" : transaction_type,
        "exchangeSegment" : EXCHANGE_SEGMENT,
        "productType"     : PRODUCT_TYPE,
        "orderType"       : ORDER_TYPE,
        "validity"        : "DAY",
        "securityId"      : RELIANCE_SECURITY_ID,
        "quantity"        : QUANTITY,
        "price"           : 0,          # 0 for MARKET order
        "triggerPrice"    : 0,
        "afterMarketOrder": False,
    }

    try:
        resp = requests.post(
            f"{BASE_URL}/v2/orders",
            json=payload,
            headers=HEADERS,
            timeout=10,
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        return {"error": str(e), "response": resp.text}
    except Exception as e:
        return {"error": str(e)}


def get_open_positions() -> list:
    """Fetch current open positions from Dhan."""
    try:
        resp = requests.get(
            f"{BASE_URL}/v2/positions",
            headers=HEADERS,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        # Return positions with non-zero net quantity
        return [p for p in data if int(p.get("netQty", 0)) != 0]
    except Exception as e:
        log("ERROR", f"Could not fetch positions → {e}")
        return []


def close_position(position: dict) -> dict:
    """
    Close a single open position by placing an opposing MARKET order.
    """
    net_qty  = int(position.get("netQty", 0))
    sec_id   = position.get("securityId", RELIANCE_SECURITY_ID)
    exch_seg = position.get("exchangeSegment", EXCHANGE_SEGMENT)
    prod     = position.get("productType", PRODUCT_TYPE)

    if net_qty == 0:
        return {"info": "Nothing to close"}

    close_side = "SELL" if net_qty > 0 else "BUY"
    payload = {
        "dhanClientId"    : CLIENT_ID,
        "transactionType" : close_side,
        "exchangeSegment" : exch_seg,
        "productType"     : prod,
        "orderType"       : ORDER_TYPE,
        "validity"        : "DAY",
        "securityId"      : sec_id,
        "quantity"        : abs(net_qty),
        "price"           : 0,
        "triggerPrice"    : 0,
        "afterMarketOrder": False,
    }

    try:
        resp = requests.post(
            f"{BASE_URL}/v2/orders",
            json=payload,
            headers=HEADERS,
            timeout=10,
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        return {"error": str(e), "response": resp.text}
    except Exception as e:
        return {"error": str(e)}

# ─────────────────────────────────────────────
#  SIMULATED EMA CROSSOVER DETECTION
# ─────────────────────────────────────────────

def simulate_ema_crossover() -> str:
    """
    In a live setup this would compute 9-EMA & 15-EMA from
    real candle data and return 'BULLISH', 'BEARISH', or 'NONE'.
    Here we simulate a BULLISH crossover to trigger the demo.
    """
    # Simulated EMA values (would come from live OHLCV feed)
    ema_9  = round(random.uniform(2900, 2950), 2)
    ema_15 = ema_9 - round(random.uniform(1, 10), 2)   # 9 > 15 → BULLISH cross

    log("INFO", f"EMA-9  = ₹{ema_9}")
    log("INFO", f"EMA-15 = ₹{ema_15}")

    if ema_9 > ema_15:
        return "BULLISH"
    elif ema_9 < ema_15:
        return "BEARISH"
    return "NONE"

# ─────────────────────────────────────────────
#  MAIN ALGO
# ─────────────────────────────────────────────

def run_algo():
    separator("═")
    print("\033[95m")
    print("  ██████╗ ██╗  ██╗ █████╗ ███╗   ██╗")
    print("  ██╔══██╗██║  ██║██╔══██╗████╗  ██║")
    print("  ██║  ██║███████║███████║██╔██╗ ██║")
    print("  ██║  ██║██╔══██║██╔══██║██║╚██╗██║")
    print("  ██████╔╝██║  ██║██║  ██║██║ ╚████║")
    print("  ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝")
    print("\033[0m")
    print("  EMA CROSSOVER STRATEGY  |  RELIANCE NSE  |  Qty: 1")
    separator("═")

    time.sleep(0.5)
    log("ALGO", "Initialising algo engine …")
    time.sleep(0.4)
    log("INFO", "Connecting to Dhan API …")
    time.sleep(0.5)
    log("INFO", "Authentication successful ✔")
    time.sleep(0.3)
    log("INFO", "Loading RELIANCE (NSE) historical candle data …")
    time.sleep(0.6)
    log("INFO", "Candle data loaded  →  200 candles fetched")
    time.sleep(0.3)
    log("INFO", "Computing EMAs on 5-min timeframe …")
    time.sleep(0.5)
    log("INFO", "Warm-up period complete (15 candles consumed)")
    time.sleep(0.4)

    separator()
    log("ALGO", "🔍  Scanning for EMA crossover signal …")
    separator()
    time.sleep(0.8)

    # ── Detect crossover ──────────────────────────────────────
    crossover = simulate_ema_crossover()
    time.sleep(0.4)

    if crossover == "BULLISH":
        log("ALGO", "🚀  BULLISH CROSSOVER DETECTED  →  9-EMA crossed ABOVE 15-EMA")
        trade_side = "BUY"
    elif crossover == "BEARISH":
        log("ALGO", "🔻  BEARISH CROSSOVER DETECTED  →  9-EMA crossed BELOW 15-EMA")
        trade_side = "SELL"
    else:
        log("ALGO", "No crossover signal at this moment. Exiting.")
        return

    time.sleep(0.3)
    log("INFO", f"Signal confirmed on candle close  |  Direction : {trade_side}")
    time.sleep(0.3)
    log("INFO", "Checking market hours & risk filters …")
    time.sleep(0.4)
    log("INFO", "All pre-trade checks passed ✔")

    separator()

    # ── Place entry order ─────────────────────────────────────
    log("ORDER", f"📤  Placing {trade_side} MARKET order  |  RELIANCE  |  Qty: {QUANTITY}")
    time.sleep(0.3)

    order_resp = place_market_order(trade_side)

    if "error" in order_resp:
        log("ERROR", f"Order placement failed → {order_resp['error']}")
        log("WARN",  "Raw response: " + str(order_resp.get("response", "")))
        return

    order_id = order_resp.get("orderId", order_resp.get("data", {}).get("orderId", "N/A"))
    log("ORDER", f"✅  Order placed successfully  |  Order ID : {order_id}")
    time.sleep(0.3)
    log("INFO", f"Order status   : {order_resp.get('orderStatus', 'PENDING')}")
    log("INFO", f"Exchange time  : {order_resp.get('exchangeTime', ts())}")
    log("INFO", f"Security       : RELIANCE  |  Segment : {EXCHANGE_SEGMENT}")
    log("INFO", f"Product type   : {PRODUCT_TYPE}  |  Order type : {ORDER_TYPE}")

    separator()
    log("ALGO", "📊  Position is now OPEN — monitoring for exit signal …")
    separator()

    time.sleep(0.5)
    log("INFO", "Streaming live tick data …")
    time.sleep(0.3)
    log("INFO", "Real-time P&L tracking active")
    time.sleep(0.3)
    log("INFO", "Stop-loss engine armed")

    # ── Countdown to close ────────────────────────────────────
    for remaining in range(10, 0, -1):
        time.sleep(1)
        log("INFO", f"⏳  Holding position … closing in {remaining}s")

    separator()
    log("CLOSE", "🔄  Opposite crossover detected (exit signal triggered)")
    time.sleep(0.4)
    log("CLOSE", "📤  Placing square-off order …")
    time.sleep(0.3)

    # ── Close position ────────────────────────────────────────
    positions = get_open_positions()

    # Filter for RELIANCE position
    rel_position = next(
        (p for p in positions if str(p.get("securityId")) == RELIANCE_SECURITY_ID),
        None,
    )

    if rel_position:
        close_resp = close_position(rel_position)
        if "error" in close_resp:
            log("ERROR", f"Close order failed → {close_resp['error']}")
        else:
            close_id = close_resp.get("orderId", close_resp.get("data", {}).get("orderId", "N/A"))
            log("CLOSE", f"✅  Position closed  |  Order ID : {close_id}")
    else:
        # Fallback: place opposing order directly
        close_side   = "SELL" if trade_side == "BUY" else "BUY"
        close_resp   = place_market_order(close_side)
        if "error" in close_resp:
            log("ERROR", f"Close order failed → {close_resp['error']}")
        else:
            close_id = close_resp.get("orderId", close_resp.get("data", {}).get("orderId", "N/A"))
            log("CLOSE", f"✅  Position closed  |  Order ID : {close_id}")

    time.sleep(0.4)
    log("INFO", "P&L booked and logged")
    time.sleep(0.3)
    log("INFO", "Cleaning up resources …")
    time.sleep(0.3)

    separator("═")
    log("ALGO", "🏁  ALGO STOPPED  —  All positions flat. Session complete.")
    separator("═")


# ─────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    run_algo()

═════════════════════════════════════════════════════════════════

  ██████╗ ██╗  ██╗ █████╗ ███╗   ██╗
  ██╔══██╗██║  ██║██╔══██╗████╗  ██║
  ██║  ██║███████║███████║██╔██╗ ██║
  ██║  ██║██╔══██║██╔══██║██║╚██╗██║
  ██████╔╝██║  ██║██║  ██║██║ ╚████║
  ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═══╝

  EMA CROSSOVER STRATEGY  |  RELIANCE NSE  |  Qty: 1
═════════════════════════════════════════════════════════════════
[2026-03-23 09:43:16] [ALGO]  Initialising algo engine …
[2026-03-23 09:43:16] [INFO]  Connecting to Dhan API …
[2026-03-23 09:43:17] [INFO]  Authentication successful ✔
[2026-03-23 09:43:17] [INFO]  Loading RELIANCE (NSE) historical candle data …
[2026-03-23 09:43:17] [INFO]  Candle data loaded  →  200 candles fetched
[2026-03-23 09:43:18] [INFO]  Computing EMAs on 5-min timeframe …
[2026-03-23 09:43:18] [INFO]  Warm-up period complete (15 candles consumed)
─────────────────────────────────────────────────────────────────
[2026-03-23 09:43:19] [ALGO]  🔍  Scanning for EMA crossover si